
**ARCH 594/508: AI Seminar/Studio**  
Spring 2026  

**Instructor:** Narjes Abbasabadi, Ph.D. Assistant Professor  
Email: nabbasab@uw.edu  

**TA:** Hasib Hossain, MSDT Studnet

Email: hasib97@uw.edu


Department of Architecture  
College of Built Environments  
University of Washington  
Website: [https://silab.be.uw.edu](https://silab.be.uw.edu)


# Introduction to Classification: Categorizing Building Energy Efficiency Levels in Seattle

This notebook will introduce the machine learning concept of classification. Using the [Seattle Energy Benchmarking Dataset](https://www.seattle.gov/environment/climate-change/buildings-and-energy/energy-benchmarking/data-and-reports), we will classify buildings based on their energy efficiency as "high" or "low", using specific building attributes and features. The distinction between "high" efficiency buildings and "low" efficiency buidings is calculated as proxy of their ENERGYSTAR score. That is, buildings with an ENERGYSTAR score lower than 50 are classified as "low" efficiency and buildings with an ENEGRYSTAR score higher than 50 are classified as "high" efficiency buildings. The classifier that we are working with in this notebook primarily requires boolean varriable inputs, thus, buildings with`'energy_efficiency_level'` denoted as `0` are low efficiency and buildings with`'energy_efficiency_level'` denoted as `1` are high efficiency.


## The Scikit Learn Machine Learning Library

In this notebook, we will work with the Scikit-Learn library, the primary library for machine learning modeling in python. Specifically, in this notebook we will explore classification.
- https://scikit-learn.org/stable/
- https://s3.amazonaws.com/assets.datacamp.com/blog_assets/Scikit_Learn_Cheat_Sheet_Python.pdf (Scikit-Learn Cheat Sheet)

![ml_map.svg](attachment:ml_map.svg)


In [ ]:
# Import necessary packages
import pandas as pd
import seaborn as sns
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import pyproj
import os
import requests
import matplotlib.ticker as ticker
import glob
import openpyxl
import geopandas as gpd
from shapely.geometry import Point
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import confusion_matrix as cm
from sklearn.tree import DecisionTreeClassifier, plot_tree

#!pip install gdown
import gdown

## We will learn how to import datasets into our google colab environment

There are many ways to do this. Some workflows include:
- Using a shareable link as we have done in class [like this second example](https://stackoverflow.com/questions/63006038/how-to-share-my-google-colaboratory-along-with-data-from-mounted-drive).
- Mounting your google drive contents as shown in [this video](https://www.youtube.com/watch?v=woHxvbBLarQ).

Alternative methods and more information can also be found [here](https://towardsdatascience.com/3-ways-to-load-csv-files-into-colab-7c14fcbdcb92).

In [ ]:
# Lets import the data from my google drive
## NOTE: MAKE SURE YOU RUN THIS CELL

# this link is to the 2021_Building_Energy_Benchmarking dataset
!gdown --id 19enMQ2D0Mv3IUq8ExyAHj8LWrKU0XFdm

# this link is to the City_Clerk_Neighborhoods.topojson
!gdown --id 1QPvtzp_ctLsQoZeIg2TDKQSvcQKd_RV6


In [ ]:
! ls # lists all of the current file directories

# IF IT WORKED '2021_Building_Energy_Benchmarking' will show up

## We will classify energy efficiency levels from structures in the Seattle building energy consumption dataset

This dataset contains building geometric and building non-geomertic characteristics for structures larger than 20,000 square feet in Seattle.

In [ ]:
# Load in the data
df= pd.read_csv('2021_Building_Energy_Benchmarking.csv')


# Preview the data
df.head()

# Lets do some data transformations for our analysis

In the following cell the distinction between "high" efficiency buildings and "low" efficiency buidings is calculated through examining each buildings `'ENERGYSTARScore'`. That is, buildings with an `'ENERGYSTARScore'` lower than 50 are classified as "low" efficiency and buildings with an `'ENERGYSTARScore'`  higher than 50 are classified as "high" efficiency buildings. The classifier that we are working with in this notebook primarily requires boolean varriable inputs, thus, buildings with`'energy_efficiency_level'` denoted as `0` are low efficiency and buildings with`'energy_efficiency_level'` denoted as `1` are high efficiency.


In [ ]:
# Make a column for converting the ENERGYSTARScore to efficiency classes
df['energy_efficiency_level'] = pd.cut(
    df['ENERGYSTARScore'],
    bins=[-1, 50,101],
    labels=[0, 1],
    right=False
)


# Then drop the ENERGYSTARScore column to remove colinearilty for analysis
df= df.drop('ENERGYSTARScore', axis=1)


# Make sure we did this correctly
df.columns

In [ ]:
# List the column attributes and data types

#df.info()

## Classification - Building Energy Efficiency Levels

In this project we will classify builings into effiency categories given building geometric features and building non-geometric features. We will classify buildings into two categories buildings with`'energy_efficiency_level'` denoted as `0` are low efficiency and buildings with`'energy_efficiency_level'` denoted as `1` are high efficiency.

## Lets first make a map of the building locations
- Visulizations based on building `'energy_efficiency_level'`.

In [ ]:
# we first need to input the Seattle geometry

shp_fn = 'City_Clerk_Neighborhoods.topojson'

seattle_gdf = gpd.read_file(shp_fn, crs='EPSG:4326')

seattle_gdf.crs = 'epsg:4326'

ax= seattle_gdf.plot()
ax.set_title("Seattle Plot");

In [ ]:
# Drop values where longitude and latitude are out of range
df = df[(df['Latitude'] != 0) & (df['Longitude'] != 0)]

# We need to convert our dataframe into a geospatial dataset
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']), crs="EPSG:4326")


## Plot the building locations on a map

- Buildings with`'energy_efficiency_level'` denoted as `0` are low efficiency and buildings with`'energy_efficiency_level'` denoted as `1` are high efficiency.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

seattle_gdf.to_crs(4326).plot(ax= ax, edgecolor='gray', facecolor='lightgray', rasterized= True);
gdf.to_crs(4326).plot(ax= ax, column='energy_efficiency_level', s=5, legend= True, legend_kwds={'loc': 'center left', 'bbox_to_anchor': (1, .85)}); #

ax.set_title('Buildings with Energy Benchmarking Data in 2021 \n colored by Energy Efficiency Level');


# Lets make visulizations of our data
Some visualization resources include:
- Seaborn plots: https://seaborn.pydata.org/tutorial/introduction.html
- Matplotlib: https://matplotlib.org/stable/plot_types/index.html


## Make a histogram of the Building Energy Efficiency Levels
- How many buildings are in each `energy_efficiency_level` categorization?


In [ ]:
# lets look at this using code
df['energy_efficiency_level'].value_counts()

In [ ]:
# now a histogram using matplotlib

plt.figure(figsize=(8, 6))
plt.hist(df['energy_efficiency_level'], bins=[-0.5, 0.5, 1.5], edgecolor='black')

plt.xticks([0, 1])
plt.xlabel('Building Energy Efficiency Level')
plt.ylabel('Number of Buildings')
plt.title('Histogram of Building Energy Efficiency Level \nin Seattle')
plt.show()

In [ ]:
# the seaborn library offers more opportunities for plot customization such as coloring by column name
# so now lets make a histogram using seaborn

plt.figure(figsize=(8, 6))
sns.histplot(data=df, x='energy_efficiency_level', hue='BuildingType', multiple='stack', bins=2, edgecolor='black')

plt.xticks([0, 1])
plt.xlabel('Building Energy Efficiency Level')
plt.ylabel('Number of Buildings')
plt.title('Histogram of Building Energy Efficiency Level \nin Seattle')

plt.show()

## 1. Split our dataset into traning and testing data

In [ ]:
# select the features that we will examine
feature_names= ['YearBuilt', 'NumberofFloors', 'PropertyGFATotal']

# transform the data to numpy
y = df['energy_efficiency_level'].to_numpy()
X = df[feature_names].to_numpy()

# generate some random indices and split the data into train/test data at the mid point
n = y.shape[0]
inds = np.arange(n)
np.random.RandomState(498)
np.random.shuffle(inds)
train_idx, test_idx = inds[:(n//2)], inds[(n//2):]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

# boolean array of which samples are equated to a high efficiency level.
high = (y == 1)
high_test = (y_test == 1)
high_train = (y_train == 1)

## 2. Run a simple classification decision tree on this dataset.

- We will use a tree depth of 3

In [ ]:
#make the model
dt_energy = DecisionTreeClassifier(max_depth=3, random_state=498)

#fit the model
dt_energy.fit(X_train, y_train)

## Lets plot the decision tree

In [ ]:
# Plot the decision tree

plt.figure(figsize=(16, 8))
plot_tree(dt_energy, feature_names=feature_names, filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree Visualization')
plt.show()


## Lets plot our model feature importances

- Model feature importances can help us determine the most influential parameters (x's) in our model

In [ ]:
# Get feature importance from the trained energy decission tree model
importances = dt_energy.feature_importances_

# Create a pandas series for better visualization
importance_df = pd.Series(importances, index=feature_names)

# Sort feature importances in descending order
importance_df = importance_df.sort_values(ascending=False)

# Plot the feature importances
plt.figure(figsize=(8,5))
plt.barh(importance_df.index, importance_df.values, color='skyblue')
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('Feature Importance in Decision Tree Classifier')
plt.gca().invert_yaxis()  # Invert y-axis to show the highest importance at the top
plt.show()


## 3. Compute our results and model accuracies in a confusion matrix

In [ ]:
# function to plot the confusion matrix

def plot_confusion_matrix(model, X, y_true, threshold=None):

  if threshold is None:
    y_pred = model.predict(X)
  else:
    y_pred = ((model.predict_proba(X)[:, 1] >= threshold) - .5) * 2 # transforming probabilities to -1 and 1.

  cm_data = cm(y_true, y_pred)
  cm_data = cm_data.ravel()[::-1].reshape(cm_data.shape)
  plt.figure(figsize=(8, 8), dpi=80)
  sns.heatmap(cm_data, annot=True, fmt=".0f", xticklabels=['Pred True', 'Pred False'],
              yticklabels = ['Actual True', 'Actual False'], cmap='coolwarm')
  return cm_data

In [ ]:
cm_data= plot_confusion_matrix(dt_energy, X_train, y_train)


## Lets calculate the precission and recall for this dataset

**Precision**: The ratio of correctly predicted positive observations to the total predicted positives.

- It answers the question: 'How many of the predicted positives are actually positive?'

`Precision = (True Positives) / (True Positives + False Positives)`

**Recall (Sensitivity)**: The ratio of correctly predicted positive observations to all observations in the actual class.

- It answers: 'How many actual positives are captured by the model?'

`Recall = (True Positives) / (True Positives + False Negative)`

In [ ]:
def print_prec_and_recall(cm_data):
  # Lets quickly calculate the precision and recall from the confusion matrix
  print("Precision: ",np.round(cm_data[0, 0]/(cm_data[0, 0]+cm_data[1, 0]),5))
  print("Recall:", np.round(cm_data[0, 0]/(cm_data[0, 0]+cm_data[0, 1]),5))
print_prec_and_recall(cm_data)